In [ ]:
    def citations_per_work(self):
        sql = """ 
        CREATE OR REPLACE TABLE memory.citations_per_work AS
            SELECT count(work_id) AS cited_by_count_endogenous,
                    sum(cited_by_count) AS cited_by_count_total,
                    author_id,
                    author_name,
                    publication_year
            FROM
                (SELECT DISTINCT id AS work_id,
                    w.cited_by_count,
                    unnest(authorships).author.id AS author_id,
                    unnest(authorships).author.display_name as author_name,
                    w.publication_year
                FROM project.raw w
                LEFT JOIN (SELECT id AS work_id,
                            unnest(referenced_works) AS cited_id
                            FROM project.raw           
                            ) c
                ON w.id = c.cited_id
                )
                GROUP BY author_id, author_name, publication_year
            ORDER BY cited_by_count_endogenous DESC
        """
        self.db.sql(sql)
        sql.db.sql("SELECT * FROM memory_citations_per_work").show()
        return

    def citations_per_work_ranked(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work 
                WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
                ORDER BY publication_year DESC, percent_rank_total DESC
                """
        self.db.sql(sql)
        return

    def citation_summation(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations AS
                SELECT author_id,
                        author_name,
                        sum(cited_by_count_total) AS citations_total,
                        sum(cited_by_count_endogenous) AS citations_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE author_id NOT NULL
                GROUP BY author_id, author_name
                ORDER BY citations_endogenous DESC                    "biblio.last_page"
                """
        self.db.sql(sql)
        return

    def hca_summation(self):       

        sql = """ 
                CREATE OR REPLACE TABLE memory.hca_endogenous AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_endogenous                    "biblio.last_page"
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_endogenous >= 0.99
                GROUP BY ALL
                ORDER BY hca_endogenous DESC;

                CREATE OR REPLACE TABLE memory.hca_total AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_total,
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_total >= 0.99
                GROUP BY ALL
                ORDER BY hca_total DESC
                """
        self.db.sql(sql)
        return
    
    def citations_endogenous_all(self):
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_endogenous_all AS
                SELECT author_id,
                        author_name,
                        citations_total,
                        citations_endogenous,
                        hca_total,
                        hca_endogenous
                FROM memory.citations
                LEFT JOIN
                    (SELECT t.*,
                            e.hca_endogenous
                        FROM memory.hca_total t
                        LEFT JOIN memory.hca_endogenous e
                        USING (author_id)
                    ) sub
                USING (author_id, author_name)
                ORDER BY citations_endogenous DESC
            """
        self.db.sql(sql)
        return

    def author_works_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.author_works_counts AS
                SELECT au.author_id,
                        au.author_name,
                        a.first,
                        a.middle,
                        a.last,
                        a.fullname,
                        a.orcid,
                        a.display_name_alternatives,
                        count(work_id) AS works_count_endogenous,
                        works_count,
                        cited_by_count,
                        "2yr_mean_citedness",
                        h_index      
                    FROM econ.authorships au
                        LEFT JOIN econ.authors a
                        ON a.author_id = au.author_id
                    GROUP BY ALL
                    ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        return
    
    def citation_summary(self):

        sql = """ 
            CREATE OR REPLACE TABLE econ.citation_summary AS
                SELECT DISTINCT c.author_id,
                        c.author_name,
                        a.author_name,
                        a.works_count_endogenous,
                        s.citations_total AS citations_total_,
                        c.citations_endogenous,
                        hca_total,
                        hca_endogenous,
                        a.orcid,
                        a.display_name_alternatives,
                        a.works_count AS works_count_total,
                        a.cited_by_count,
                        a."2yr_mean_citedness",                
                SetUp:

    def __init__(self):
        self._setup_db()
        return

        self.db.sql("SHOW ALL TABLES").show()
        return
                        a.h_index
                FROM memory.citations c
                    LEFT JOIN memory.citations_endogenous_all s
                    ON c.author_id = s.author_id
                        LEFT JOIN memory.author_works_counts a
                        ON c.author_id = a.author_id
            ORDER BY cited_by_count DESC, h_index DESC
            """
        self.db.sql(sql)
        return
    
    def show_all(self):
        self.db.sql("SELECT * FROM memory.citations_per_work").show()
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked").show()
        self.db.sql("SELECT * FROM memory.citations").show()
        self.db.sql("SELECT * FROM memory.hca_endogenous").show() 
        self.db.sql("SELECT * FROM memory.citations_endogenous_all").show()
        self.db.sql("SELECT * FROM econ.authors").show()  
        self.db.sql("SELECT * FROM econ.citation_summary").show()
        return
    
    def load_citations(self):
        df = self.db.sql("""
                         SELECT * EXCLUDE (author_name_1, display_name_alternatives) FROM project.citation_summary ORDER BY hca_endogenous DESC, h_index DESC
                         """).df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx', index=False)
        return
